# Data understanding

**Rubric: Data Understanding (5 pts).** Business problem, dataset and source, relevant variables, data-quality checks, limitations.

- What's in the snapshot, what we removed and why (`filters`)
- Missing values, odd values, the 22 vs 23 neighbourhood count
- Is our revenue estimate independent? (`valid`: it isn't, it matches Inside Airbnb's formula)

In [8]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `raw`: listings before cleaning
- `df`: cleaned listings (one row per listing, with our revenue columns)
- `filters`: rows removed by each cleaning step
- `valid`: our revenue model vs Inside Airbnb's estimate

In [9]:
raw = pd.read_csv(config.RAW / "listings.csv.gz", low_memory=False)  # before cleaning
df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)  # cleaned listings
filters = pd.read_csv(T / "filters_applied.csv")
valid = pd.read_csv(T / "revenue_validation.csv")

In [10]:
cols = ["id", "neighbourhood_cleansed", "room_type", "bucket", "bedrooms", "price_num",
        "number_of_reviews_ltm", "is_active", "rev_review_model"]
small = df[cols]
small.head()

,id,neighbourhood_cleansed,room_type,bucket,bedrooms,price_num,number_of_reviews_ltm,is_active,rev_review_model
0,13126,Shaughnessy,Private room,other,1.00,425.00,4,True,"10,200.00"
1,13188,Riley Park,Entire home/apt,suite,NaN,487.00,36,True,"105,192.00"
2,13358,Downtown,Entire home/apt,condo,1.00,429.00,33,True,"84,942.00"
3,16395,Downtown Eastside,Entire home/apt,condo,1.00,467.00,0,False,0.00
4,18589,Grandview-Woodland,Private room,house,NaN,286.00,47,True,"73,073.00"


### Data-quality problems (selected columns)
| Problem | How big | What we do |
|---|---|---|
| `bedrooms` is missing | 378 listings (9%) | Assume 1 bedroom (likely studios) |
| No reviews in the last year | 15% of listings | Revenue = $0, marked not active |
| Price is the World Cup week quote | All listings | Can't fix; name it as a risk |
| Revenue is an estimate, not real | All listings | Same formula as Inside Airbnb, so not a real cross-check |

Other columns have no missing values. Prices: $64 to $1,974 a night, typical $382.

## Quick look

In [11]:
filters

,step,rows_before,removed,why
0,has_price,6509,664,no price: not bookable at scrape
1,price_in_range,5845,399,outside $30-$2000/night
2,short_term_only,5446,1045,"min stay >= 30 nights = monthly rental, not STR"


In [12]:
valid

,method,p25,median,p75,n
0,rev_review_model,"13,416.00","37,230.78","69,539.43","3,737.00"
1,rev_iab_estimate,"13,416.00","37,231.00","69,470.00","3,737.00"
2,spearman_corr,NaN,1.00,NaN,NaN


In [13]:
raw.isna().mean().sort_values(ascending=False).head(15)

host_verifications             1.00
host_since                     1.00
neighbourhood_group_cleansed   1.00
neighbourhood                  1.00
host_total_listings_count      1.00
host_neighbourhood             1.00
host_thumbnail_url             1.00
host_acceptance_rate           1.00
host_response_rate             1.00
host_response_time             1.00
calendar_updated               1.00
neighborhood_overview          1.00
instant_bookable               1.00
host_about                     0.50
bathrooms                      0.38
dtype: float64

## Analysis
_Start here._

In [16]:
small.head(15)

,id,neighbourhood_cleansed,room_type,bucket,bedrooms,price_num,number_of_reviews_ltm,is_active,rev_review_model
0,13126,Shaughnessy,Private room,other,1.00,425.00,4,True,"10,200.00"
1,13188,Riley Park,Entire home/apt,suite,NaN,487.00,36,True,"105,192.00"
2,13358,Downtown,Entire home/apt,condo,1.00,429.00,33,True,"84,942.00"
3,16395,Downtown Eastside,Entire home/apt,condo,1.00,467.00,0,False,0.00
4,18589,Grandview-Woodland,Private room,house,NaN,286.00,47,True,"73,073.00"
5,33150,Riley Park,Entire home/apt,condo,1.00,194.00,19,True,"36,860.00"
6,35304,Grandview-Woodland,Entire home/apt,house,1.00,227.50,42,True,"57,330.00"
7,77157,Arbutus Ridge,Entire home/apt,suite,1.00,328.50,44,True,"83,931.75"
8,140698,Mount Pleasant,Private room,house,NaN,207.00,31,True,"52,888.50"
9,254657,Downtown Eastside,Entire home/apt,condo,1.00,429.67,15,True,"38,670.30"
